# Soft Cube Goal-Pose RL — NVIDIA Warp + PyTorch

GPU-parallel version of the non-Warp task.

Goal:
- COM target `(0.55, 0.25, SIDE/2)`
- target orientation `+60° yaw`
- orientation estimated by batched Kabsch fitting

Physics structure is deliberately close to `real2sim-eval`:
1. clear forces
2. spring-parallel force evaluation
3. atomic force accumulation
4. velocity update with gravity + exponential drag
5. restitution/friction table collision
6. PyTorch pose/reward/PPO on the same GPU

In [ ]:
!nvidia-smi
%pip -q install "warp-lang==1.17.0" torch numpy matplotlib

import warp as wp
import torch, numpy as np, math
wp.init()
DEVICE="cuda:0" if wp.is_cuda_available() else "cpu"
TORCH_DEVICE=torch.device("cuda" if DEVICE.startswith("cuda") else "cpu")
print("Warp device:", DEVICE)

In [ ]:
SIDE=0.40
def cube_topology(side=SIDE,z0=0.22):
    X=np.array([
        [-1,-1,-1],[ 1,-1,-1],[-1, 1,-1],[ 1, 1,-1],
        [-1,-1, 1],[ 1,-1, 1],[-1, 1, 1],[ 1, 1, 1],
    ],dtype=np.float32)*(side/2)
    X[:,2]+=z0
    springs=[]; act=[]; e=0
    for i in range(8):
        for j in range(i+1,8):
            springs.append((i,j))
            L=np.linalg.norm(X[j]-X[i])
            if np.isclose(L,side,atol=1e-5):
                act.append(e); e+=1
            else: act.append(-1)
    si=np.array([i for i,j in springs],np.int32)
    sj=np.array([j for i,j in springs],np.int32)
    L0=np.linalg.norm(X[sj]-X[si],axis=1).astype(np.float32)
    return X,si,sj,L0,np.array(act,np.int32)

X_REF,SI,SJ,L0,ACT_ID=cube_topology()
N_ACT=int(np.sum(ACT_ID>=0))
REF_CENTERED=X_REF-X_REF.mean(axis=0,keepdims=True)

REF_CENTERED_T=torch.tensor(REF_CENTERED,dtype=torch.float32,device=TORCH_DEVICE)
GOAL_POS_T=torch.tensor([0.55,0.25,SIDE/2],dtype=torch.float32,device=TORCH_DEVICE)
yaw=math.radians(60.0)
GOAL_R_T=torch.tensor([
    [math.cos(yaw),-math.sin(yaw),0],
    [math.sin(yaw), math.cos(yaw),0],
    [0,0,1]],dtype=torch.float32,device=TORCH_DEVICE)

def kabsch_batch(x):
    com=x.mean(dim=1,keepdim=True)
    B=x-com
    A=REF_CENTERED_T.unsqueeze(0).expand(x.shape[0],-1,-1)
    H=A.transpose(1,2)@B
    U,S,Vh=torch.linalg.svd(H)
    V=Vh.transpose(1,2)
    R=V@U.transpose(1,2)
    det=torch.det(R)
    Vfix=V.clone()
    Vfix[det<0,:,-1]*=-1
    R=Vfix@U.transpose(1,2)
    return com[:,0,:],R

def rotation_angle_batch(R,Rg):
    E=R.shape[0]
    Rerr=Rg.expand(E,-1,-1).transpose(1,2)@R
    tr=Rerr[:,0,0]+Rerr[:,1,1]+Rerr[:,2,2]
    c=torch.clamp((tr-1.0)/2.0,-1.0,1.0)
    return torch.acos(c)

print("particles",len(X_REF),"springs",len(SI),"actuators",N_ACT)

## Warp physics kernels

In [ ]:
@wp.kernel
def reset_kernel(base_x:wp.array(dtype=wp.vec3),
                 x:wp.array(dtype=wp.vec3),
                 v:wp.array(dtype=wp.vec3),
                 force:wp.array(dtype=wp.vec3)):
    tid=wp.tid(); p=tid%8
    x[tid]=base_x[p]
    v[tid]=wp.vec3(0.0,0.0,0.0)
    force[tid]=wp.vec3(0.0,0.0,0.0)

@wp.kernel
def clear_force_kernel(force:wp.array(dtype=wp.vec3)):
    force[wp.tid()]=wp.vec3(0.0,0.0,0.0)

@wp.kernel
def spring_kernel(
    x:wp.array(dtype=wp.vec3), v:wp.array(dtype=wp.vec3),
    si:wp.array(dtype=wp.int32), sj:wp.array(dtype=wp.int32),
    rest:wp.array(dtype=float), act_id:wp.array(dtype=wp.int32),
    actions:wp.array2d(dtype=float), force:wp.array(dtype=wp.vec3),
    n_springs:int, stiffness:float, dashpot:float, act_amp:float):
    tid=wp.tid()
    env=tid//n_springs
    s=tid-env*n_springs
    base=env*8
    i=si[s]; j=sj[s]
    x1=x[base+i]; x2=x[base+j]
    v1=v[base+i]; v2=v[base+j]
    dvec=x2-x1
    L=wp.length(dvec)
    n=dvec/wp.max(L,1.0e-8)
    L0s=rest[s]
    aid=act_id[s]
    if aid>=0:
        L0s=L0s*(1.0+act_amp*actions[env,aid])

    spring_mag=stiffness*(L/L0s-1.0)
    vrel=wp.dot(v2-v1,n)
    f=(spring_mag+dashpot*vrel)*n
    wp.atomic_add(force,base+i,f)
    wp.atomic_sub(force,base+j,f)

@wp.kernel
def update_velocity_kernel(
    v:wp.array(dtype=wp.vec3), force:wp.array(dtype=wp.vec3),
    mass:float, dt:float, drag:float):
    tid=wp.tid()
    g=wp.vec3(0.0,0.0,-9.81*mass)
    vv=v[tid]+(force[tid]+g)/mass*dt
    v[tid]=vv*wp.exp(-dt*drag)

@wp.kernel
def table_contact_integrate_kernel(
    x:wp.array(dtype=wp.vec3), v:wp.array(dtype=wp.vec3),
    dt:float, restitution:float, friction:float):
    tid=wp.tid()
    x0=x[tid]; v0=v[tid]
    z=x0[2]; vz=v0[2]
    znext=z+vz*dt

    v1=v0
    toi=0.0
    if znext<0.0 and vz<-1.0e-6:
        normal=wp.vec3(0.0,0.0,1.0)
        vn=wp.dot(v0,normal)*normal
        vt=v0-vn
        vn_len=wp.length(vn)
        vt_len=wp.max(wp.length(vt),1.0e-8)
        vn_new=-restitution*vn
        a=wp.max(0.0,1.0-friction*(1.0+restitution)*vn_len/vt_len)
        v1=vn_new+a*vt
        if z>0.0:
            toi=wp.clamp(-z/vz,0.0,dt)

    x1=x0+v0*toi+v1*(dt-toi)
    if x1[2]<0.0:
        x1[2]=0.0
        if v1[2]<0.0:
            v1[2]=0.0
    x[tid]=x1
    v[tid]=v1

## Vectorized Warp environment

In [ ]:
SI_T=torch.tensor(SI,dtype=torch.long,device=TORCH_DEVICE)
SJ_T=torch.tensor(SJ,dtype=torch.long,device=TORCH_DEVICE)
L0_T=torch.tensor(L0,dtype=torch.float32,device=TORCH_DEVICE)

class WarpGoalPoseEnv:
    def __init__(self,num_envs=1024,dt=0.0025,substeps=4,mass=0.15,
                 stiffness=72.0,dashpot=2.0,drag=0.5,act_amp=0.20,
                 restitution=0.15,friction=0.65):
        self.num_envs=num_envs; self.dt=dt; self.substeps=substeps
        self.mass=mass; self.stiffness=stiffness; self.dashpot=dashpot
        self.drag=drag; self.act_amp=act_amp
        self.restitution=restitution; self.friction=friction
        self.n_springs=len(SI); self.n_act=N_ACT

        self.base_x=wp.array(X_REF,dtype=wp.vec3,device=DEVICE)
        self.si=wp.array(SI,dtype=wp.int32,device=DEVICE)
        self.sj=wp.array(SJ,dtype=wp.int32,device=DEVICE)
        self.rest=wp.array(L0,dtype=float,device=DEVICE)
        self.act_id=wp.array(ACT_ID,dtype=wp.int32,device=DEVICE)
        N=num_envs*8
        self.x=wp.zeros(N,dtype=wp.vec3,device=DEVICE)
        self.v=wp.zeros(N,dtype=wp.vec3,device=DEVICE)
        self.force=wp.zeros(N,dtype=wp.vec3,device=DEVICE)
        self.reset()

    def reset(self):
        wp.launch(reset_kernel,dim=self.num_envs*8,
                  inputs=[self.base_x,self.x,self.v,self.force],device=DEVICE)
        self.success_count=torch.zeros(self.num_envs,dtype=torch.int32,device=TORCH_DEVICE)
        dp,da,_,_=self.errors()
        self.prev_dp=dp.clone(); self.prev_da=da.clone()
        return self.observation()

    def state(self):
        x=wp.to_torch(self.x).view(self.num_envs,8,3)
        v=wp.to_torch(self.v).view(self.num_envs,8,3)
        return x,v

    def errors(self):
        x,_=self.state()
        p,R=kabsch_batch(x)
        dp=torch.linalg.norm(GOAL_POS_T[None,:]-p,dim=1)
        da=rotation_angle_batch(R,GOAL_R_T[None,:,:])
        return dp,da,p,R

    def observation(self):
        x,v=self.state()
        p,R=kabsch_batch(x)
        rel=x-p[:,None,:]
        goal_delta=GOAL_POS_T[None,:]-p
        Rerr=GOAL_R_T.T[None,:,:]@R
        return torch.cat([rel.reshape(self.num_envs,-1),v.reshape(self.num_envs,-1),
                          goal_delta,Rerr.reshape(self.num_envs,-1)],dim=1)

    def step(self,actions):
        actions=actions.contiguous().float()
        aw=wp.from_torch(actions,requires_grad=False)

        for _ in range(self.substeps):
            wp.launch(clear_force_kernel,dim=self.num_envs*8,
                      inputs=[self.force],device=DEVICE)
            wp.launch(spring_kernel,dim=self.num_envs*self.n_springs,
                      inputs=[self.x,self.v,self.si,self.sj,self.rest,self.act_id,
                              aw,self.force,self.n_springs,self.stiffness,
                              self.dashpot,self.act_amp],device=DEVICE)
            wp.launch(update_velocity_kernel,dim=self.num_envs*8,
                      inputs=[self.v,self.force,self.mass,self.dt,self.drag],device=DEVICE)
            wp.launch(table_contact_integrate_kernel,dim=self.num_envs*8,
                      inputs=[self.x,self.v,self.dt,self.restitution,self.friction],
                      device=DEVICE)

        dp,da,p,R=self.errors()
        x,_=self.state()
        curL=torch.linalg.norm(x[:,SJ_T]-x[:,SI_T],dim=-1)
        deform=((curL-L0_T[None,:])/L0_T[None,:]).square().mean(dim=1)

        reward=(5.0*(self.prev_dp-dp)+1.0*(self.prev_da-da)
                -0.002*actions.square().mean(dim=1)-0.01*deform)

        ok=(dp<0.05)&(da<math.radians(15.0))
        self.success_count=torch.where(ok,self.success_count+1,torch.zeros_like(self.success_count))
        success=self.success_count>=20
        reward=reward+0.02*ok.float()+5.0*success.float()

        self.prev_dp=dp.clone(); self.prev_da=da.clone()
        return self.observation(),reward,{
            "pos_error":dp,"angle_error":da,"com":p,"success":success}

env=WarpGoalPoseEnv(num_envs=256)
obs=env.reset()
a=torch.zeros((256,N_ACT),device=TORCH_DEVICE)
obs,r,info=env.step(a)
print("obs",obs.shape,"reward",r.shape)
print("mean position error",float(info["pos_error"].mean()))
print("mean angle error deg",float(torch.rad2deg(info["angle_error"]).mean()))

## PPO

In [ ]:
import torch.nn as nn
from torch.distributions import Normal

class ActorCritic(nn.Module):
    def __init__(self,obs_dim=60,act_dim=N_ACT,h=128):
        super().__init__()
        self.actor=nn.Sequential(nn.Linear(obs_dim,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,act_dim))
        self.critic=nn.Sequential(nn.Linear(obs_dim,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,1))
        self.log_std=nn.Parameter(torch.full((act_dim,),-0.5))
    def dist(self,o):
        mu=self.actor(o); return Normal(mu,self.log_std.exp().expand_as(mu))
    def value(self,o): return self.critic(o).squeeze(-1)
    @torch.no_grad()
    def sample(self,o):
        d=self.dist(o); z=d.sample(); a=torch.tanh(z)
        lp=d.log_prob(z).sum(-1)-torch.log(1-a*a+1e-6).sum(-1)
        return a,z,lp,self.value(o)
    def evaluate(self,o,z):
        d=self.dist(o); a=torch.tanh(z)
        lp=d.log_prob(z).sum(-1)-torch.log(1-a*a+1e-6).sum(-1)
        return lp,d.entropy().sum(-1),self.value(o)

@torch.no_grad()
def gae_vec(R,V,lastV,gamma=.99,lam=.95):
    T,E=R.shape; A=torch.zeros_like(R); g=torch.zeros(E,device=R.device)
    for t in reversed(range(T)):
        nv=lastV if t==T-1 else V[t+1]
        delta=R[t]+gamma*nv-V[t]
        g=delta+gamma*lam*g; A[t]=g
    return A,A+V

def train_warp_ppo(num_envs=1024,updates=30,horizon=32,ppo_epochs=4,minibatch=4096):
    env=WarpGoalPoseEnv(num_envs=num_envs)
    net=ActorCritic().to(TORCH_DEVICE)
    opt=torch.optim.Adam(net.parameters(),lr=3e-4)
    hist={"reward":[],"pos_error":[],"angle_deg":[]}

    for upd in range(updates):
        obs=env.reset()
        O=[];Z=[];LP=[];RR=[];VV=[]
        for _ in range(horizon):
            a,z,lp,val=net.sample(obs)
            nxt,r,info=env.step(a)
            O.append(obs);Z.append(z);LP.append(lp);RR.append(r);VV.append(val)
            obs=nxt

        with torch.no_grad(): lastV=net.value(obs)
        O=torch.stack(O);Z=torch.stack(Z);OLDLP=torch.stack(LP)
        R=torch.stack(RR);V=torch.stack(VV)
        ADV,RET=gae_vec(R,V,lastV)
        ADV=(ADV-ADV.mean())/(ADV.std()+1e-8)

        O=O.reshape(-1,60);Z=Z.reshape(-1,N_ACT)
        OLDLP=OLDLP.reshape(-1);ADV=ADV.reshape(-1);RET=RET.reshape(-1)
        B=O.shape[0]

        for _ in range(ppo_epochs):
            perm=torch.randperm(B,device=TORCH_DEVICE)
            for st in range(0,B,minibatch):
                b=perm[st:st+minibatch]
                lp,ent,val=net.evaluate(O[b],Z[b])
                ratio=(lp-OLDLP[b]).exp()
                ploss=-torch.minimum(ratio*ADV[b],torch.clamp(ratio,.8,1.2)*ADV[b]).mean()
                loss=ploss+.5*(val-RET[b]).square().mean()-.002*ent.mean()
                opt.zero_grad(set_to_none=True);loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(),.5);opt.step()

        hist["reward"].append(float(R.mean()))
        hist["pos_error"].append(float(info["pos_error"].mean()))
        hist["angle_deg"].append(float(torch.rad2deg(info["angle_error"]).mean()))
        print(f"{upd+1:03d} r={hist['reward'][-1]:+.5f} "
              f"pos={hist['pos_error'][-1]:.3f}m rot={hist['angle_deg'][-1]:.1f}deg")
    return net,hist

# smoke:
# policy,hist=train_warp_ppo(num_envs=256,updates=3,horizon=16,ppo_epochs=2,minibatch=1024)

# longer:
# policy,hist=train_warp_ppo(num_envs=1024,updates=50,horizon=32)

## Suggested comparison experiments
1. Position-only vs position+orientation reward.
2. 28-link cube vs radius-neighbor point-cloud topology.
3. Dashpot-only vs dashpot + global drag.
4. Penalty contact vs restitution/friction impulse contact.
5. Randomized target pose for a general goal-conditioned controller.